# Day 42 — Solutions

## Naive Bayes

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | Counting and Bayes' rule | 1 – 4 |
| **B** | The naive assumption | 5 – 7 |
| **C** | Smoothing and logs | 8 – 10 |
| **D** | Choosing a flavour | 11 – 12 |

Run the setup cell first. Every question is self-contained after that.

> This is the solutions file. Each question shows working code and then the
> reasoning. Read the reasoning — most of the marks are there.

In [ ]:
import math, time, warnings
import numpy as np
warnings.simplefilter("ignore")

from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.datasets import load_digits, load_breast_cancer, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

CV = StratifiedKFold(5, shuffle=True, random_state=0)

# ---- a NEW corpus, different from the one in the teaching notebook ---------
# 24 product reviews. 1 = a complaint, 0 = a compliment.
BAD = [
    "the battery died after one week",
    "screen cracked on day two",
    "stopped working after a month",
    "the battery is terrible",
    "broke immediately do not buy",
    "poor quality and slow delivery",
    "screen is dim and the battery drains",
    "it stopped charging completely",
    "cheap plastic feels fragile",
    "returned it the same week",
]
GOOD = [
    "excellent battery life so far",
    "fast delivery and well packed",
    "works perfectly every day",
    "great screen and sharp colours",
    "lovely build quality",
    "exactly what i wanted",
    "the battery lasts all day",
    "arrived early and works well",
    "very happy with this",
    "sturdy and reliable so far",
    "clear screen and easy setup",
    "good value for the price",
    "comfortable and light",
    "does the job nicely",
]

REVIEWS = [(r, 1) for r in BAD] + [(r, 0) for r in GOOD]   # 1 = complaint

def has(doc, word):
    return word in doc.split()

def count(word, label=None):
    return sum(1 for d, y in REVIEWS if has(d, word) and (label is None or y == label))

def n(label=None):
    return sum(1 for _, y in REVIEWS if label is None or y == label)

print("reviews:", n(), " complaints:", n(1), " compliments:", n(0))
for w in ["battery", "screen", "works"]:
    print(f'  "{w}" appears in {count(w)} reviews '
          f'({count(w,1)} complaint, {count(w,0)} compliment)')

---
# Part A · Counting and Bayes' rule

## Question 1

A new review contains the word **battery**. Work out `P(complaint | battery)` the way
we did in class — *without any formula*. Print every review containing "battery", then
the fraction of those that are complaints.

In [ ]:
survivors = [(d, y) for d, y in REVIEWS if has(d, "battery")]

for d, y in survivors:
    print(f"   {'COMPLAINT ' if y else 'compliment'}   {d}")

bad = sum(y for _, y in survivors)
print()
print(f"{len(survivors)} reviews mention 'battery', of which {bad} are complaints")
print(f"P(complaint | battery) = {bad}/{len(survivors)} = {bad/len(survivors):.4f}")

### Answer

Five reviews mention "battery" and **three** of them are complaints, so
`P(complaint | battery) = 3/5 = 0.6000`.

Now compare that with the base rate: 10 of the 24 reviews are complaints, so before
reading any words at all the answer would have been 0.4167. Seeing "battery" moved it
to 0.60 — a real shift, but nothing like decisive. "Battery" is a word people reach for
when the battery is *good* as well as bad. Contrast it with "free" in the lecture
corpus, which took 0.40 all the way to 6/7 = 0.857.

A clue is useful exactly to the extent that it appears at *different* rates in the two
classes. One that is equally common in both tells you nothing, however often it turns
up.

## Question 2

Now get the same number out of Bayes' rule. Compute the prior, the likelihood and the
evidence separately, combine them, and check the result against Question 1.

$$P(\text{complaint} \mid \text{battery}) =
\frac{P(\text{battery} \mid \text{complaint}) \times P(\text{complaint})}{P(\text{battery})}$$

In [ ]:
prior      = n(1) / n()
likelihood = count("battery", 1) / n(1)
evidence   = count("battery") / n()
posterior  = likelihood * prior / evidence

print(f"prior       P(complaint)            = {n(1)}/{n()}  = {prior:.4f}")
print(f"likelihood  P(battery | complaint)  = {count('battery',1)}/{n(1)}  = {likelihood:.4f}")
print(f"evidence    P(battery)              = {count('battery')}/{n()}  = {evidence:.4f}")
print(f"posterior                           = {posterior:.4f}")
print()
print("matches Question 1:",
      abs(posterior - count("battery", 1) / count("battery")) < 1e-12)

### Answer

| | | |
|---|---|---|
| prior | `P(complaint)` = 10/24 | 0.4167 |
| likelihood | `P(battery \| complaint)` = 3/10 | 0.3000 |
| evidence | `P(battery)` = 5/24 | 0.2083 |

`0.3000 × 0.4167 ÷ 0.2083 = 0.6000` — identical to the count, as it must be.

Bayes' rule did not add any information here, and it cannot: it is an identity, a
rearrangement of the same counts. What it does is let you reach the answer from the
*likelihood* direction, which matters enormously once the table is too big to cover
with your hand — and that happens as soon as you have more than one or two clues.

Notice the journey: the prior was 0.4167 and the evidence lifted it to 0.6000. The word
helped, but it did not transform the picture. Most single clues are like this; the
power of naive Bayes comes from multiplying many weak clues together, not from any one
of them being decisive.

## Question 3

Here are two statements about this corpus:

- **(i)** `P(battery | complaint)` = 0.30
- **(ii)** `P(complaint | battery)` = 0.60

Both are true and they are different numbers. Explain in your own words what each one
is actually asking, and describe a situation where confusing the two would lead
somebody to a badly wrong conclusion.

### Answer

**(i) `P(battery | complaint)` = 0.30** starts from the complaints and looks at their
words: *of the ten complaints, three mention the battery.* The population being
examined is the 10 complaints.

**(ii) `P(complaint | battery)` = 0.60** starts from the word and looks at the labels:
*of the five reviews mentioning the battery, three are complaints.* The population is
the 5 battery reviews.

They are different because **the two populations have different sizes** — 10 versus 5.
The numerator is the same 3 in both cases; only the denominator changes.

**Where confusing them goes wrong.** The classic case is medical testing. Suppose a
test for a rare disease is 99% accurate, meaning `P(positive | disease) = 0.99`. A
patient tests positive and is told they almost certainly have the disease. But that is
`P(disease | positive)`, which is a completely different quantity — and if the disease
affects 1 person in 10,000, the great majority of positive results are false alarms
from the enormous healthy population. The true answer can be under 1%.

The same error has sent people to prison. A prosecutor says "the chance of this DNA
match occurring by chance is one in a million" — that is
`P(match | innocent)` — and the jury hears it as "the chance the defendant is innocent
is one in a million", which is `P(innocent | match)`. In a country of 70 million those
are wildly different. This is why the mistake is called the **prosecutor's fallacy**.

## Question 4

The word **screen** appears in both classes. Work out `P(complaint | screen)` and
compare it with the prior `P(complaint)`.

Then answer: is "screen" a useful clue? What would make a clue useless, and what would
make one perfect?

In [ ]:
prior = n(1) / n()
for w in ["battery", "screen", "works", "and"]:
    p = count(w, 1) / count(w)
    shift = p - prior
    print(f"{w:>9}  appears {count(w):>2} times   "
          f"P(complaint | {w}) = {p:.4f}   shift {shift:+.4f}")

print()
print(f"prior P(complaint) = {prior:.4f}")

### Answer

| word | appears | P(complaint \| word) | shift from the prior (0.4167) |
|---|---|---|---|
| battery | 5 | 0.6000 | +0.1833 |
| screen | 4 | 0.5000 | +0.0833 |
| works | 2 | 0.0000 | −0.4167 |
| and | 8 | 0.2500 | −0.1667 |

**"Screen" is the weakest real clue here.** It lifts the probability from 0.4167 to
0.5000 — it moves the answer, but only just.

**In principle, a useless clue appears at the same rate in both classes**, leaving the
probability exactly at the prior. **A perfect clue appears in one class and never in
the other** — "works" is one: two appearances, both in compliments, giving
`P(complaint | works) = 0.0000`.

**But look carefully at "and".** It is the most common word in the corpus, it carries no
meaning whatsoever, and it still shifts the answer by −0.1667 — a bigger shift than
"screen", a word that genuinely is about the product. That shift is **pure accident**:
with 24 reviews, a meaningless stop-word can easily land 8 times in a lopsided way.

That is the real lesson of this question. At this sample size you cannot distinguish a
weak clue from a lucky one, and frequency is not usefulness. It is why stop-words like
"and", "the" and "is" are normally stripped out before modelling, and why the apparent
informativeness of any single word on a small corpus should be treated with suspicion.

And notice the danger in the "works" row: a perfect clue is also a **zero**, which is
what Part C is about. A clue that looks perfect on 24 reviews is usually just one you
have not seen enough of.

---
# Part B · The naive assumption

## Question 5

Count how many **separate probabilities** you would have to estimate to do the honest
(non-naive) calculation, as the number of words you look at grows.

Print a table for 1, 2, 3, 5, 10 and 20 words, and compare each to the 24 reviews you
actually have.

In [ ]:
print(f"{'words':>6}  {'combinations':>14}  {'reviews per combination':>24}")
for k in [1, 2, 3, 5, 10, 20]:
    combos = 2 ** k
    print(f"{k:>6}  {combos:>14,}  {n()/combos:>24.6f}")

### Answer

Each word is present or absent, so `k` words make `2**k` combinations — and the table
has to be filled *for each class*.

The third column is the damning one. At 5 words there is less than one review per
combination; by 10 words it is 0.023, meaning 97.7% of the boxes are empty. An empty
box gives a probability of 0/0 — not a small number, no number at all.

And 20 words is still a *tiny* message. A single real email has hundreds.

This is why "just collect more data" does not rescue the honest calculation: the
requirement doubles with every word while any real dataset grows linearly at best. The
naive assumption is not laziness, it is the only way the problem is tractable at all.

## Question 6

Measure how wrong the naive assumption is on **this** corpus, for the words
**battery** and **screen**, within the complaints.

Compare the exact joint probability with the product of the two separate ones.

In [ ]:
def joint(w1, w2, label):
    return sum(1 for d, y in REVIEWS if y == label and has(d, w1) and has(d, w2))

exact = joint("battery", "screen", 1) / n(1)
naive = (count("battery", 1) / n(1)) * (count("screen", 1) / n(1))

print(f"complaints containing both  : {joint('battery','screen',1)} of {n(1)}")
print(f"exact P(battery and screen | complaint) = {exact:.4f}")
print(f"naive P(battery|c) x P(screen|c)        = {naive:.4f}")
print(f"ratio exact/naive                       = {exact/naive:.3f}")
print()
for d, y in REVIEWS:
    if y == 1 and has(d, "battery") and has(d, "screen"):
        print("   the review with both:", repr(d))

### Answer

Exact `1/10 = 0.1000`; naive `0.3000 × 0.2000 = 0.0600`. The ratio is **1.667**.

So independence is wrong here too — reality has about **two thirds more** joint
occurrences than the assumption allows — though less badly wrong than the factor of
**2** we measured in the lecture corpus. The size of the error depends entirely on how
strongly the two clues actually travel together.

The review responsible is *"screen is dim and the battery drains"* — someone listing
two faults in one sentence, which is exactly the kind of correlation the assumption
denies.

The honest summary is that the naive assumption's error is **not a fixed quantity**.
It is small when clues are nearly unrelated and large when they are not, and you
generally cannot know in advance which case you are in. What you *can* do is what Part
D does: just measure whether the model works.

## Question 7

The assumption is false — you just measured it, twice, on two different corpora. Yet
naive Bayes remains one of the most widely deployed classifiers ever written.

Explain why being wrong about the probabilities does not necessarily make it wrong
about the **labels**. Then say what this implies about using `predict_proba()`.

### Answer

**Because classifying only needs a comparison, not a quantity.** To label a review the
model computes a score for "complaint" and a score for "compliment" and returns
whichever is larger. The actual size of those scores is thrown away. So the assumption
can distort both numbers badly and the model still gets the right answer, provided it
distorts them in a way that leaves the **order** intact.

That is usually what happens. Correlated clues act like double-counting: if "battery"
and "screen" tend to appear together, counting both is really counting one piece of
evidence twice. But the double-counting inflates the score of whichever class that
evidence already favoured — so the winner is pushed further ahead rather than
overtaken.

**What it implies about `predict_proba()`.** Those same inflated scores *are* the
reported probabilities, so the confidences come out pushed hard towards 0 and 1. Naive
Bayes will tell you it is 99.9% sure far more often than it is actually right 999 times
in 1000. It is a well-calibrated *ranker* and a badly calibrated *estimator*.

So: use `predict()` freely. Do not quote `predict_proba()` as a real probability, and
do not use it to set a decision threshold or compute an expected cost without
calibrating it first (sklearn has `CalibratedClassifierCV` for exactly this).

---
# Part C · Smoothing and logs

## Question 8

Classify the review **"the battery works"** with **no smoothing**, by hand.

Write a `score` function that multiplies the prior by one likelihood per word, run it
for both classes, and print the resulting probability of "complaint".

Something will go wrong. Say what, and why.

In [ ]:
def score(words, label, alpha=0.0):
    p = n(label) / n()
    for w in words:
        p *= (count(w, label) + alpha) / (n(label) + 2 * alpha)
    return p

words = ["the", "battery", "works"]
s_bad, s_good = score(words, 1), score(words, 0)

for w in words:
    print(f"  P({w:>8} | complaint) = {count(w,1)}/{n(1)} = {count(w,1)/n(1):.4f}")
print()
print(f"complaint score  {s_bad:.8f}")
print(f"compliment score {s_good:.8f}")
print(f"P(complaint)     {s_bad/(s_bad+s_good):.4f}")

### Answer

The complaint score is **exactly 0.0**, so `P(complaint) = 0.0000`.

The culprit is `P(works | complaint) = 0/10 = 0`. The word "works" never appears in any
of the ten complaints, and **anything multiplied by zero is zero** — so that single
word annihilates the whole product regardless of what the other words say.

Two things are wrong with that:

1. **It is an overconfident claim.** The model is asserting it is *impossible* for a
   complaint to contain "works". But "it never works" and "stopped working" are obvious
   complaints. We simply have not seen one in a corpus of 24.
2. **It cannot be outvoted.** Multiplication by zero is absorbing. A thousand words of
   furious complaint would not rescue the score.

This is the zero-frequency problem, and it is not an edge case — with real vocabularies
most words are unseen in most classes, so without a fix the model would return 0 for
almost everything.

## Question 9

Repair it with Laplace smoothing. Sweep `alpha` over
`[0, 0.01, 0.1, 0.5, 1, 5, 20]` and print, for each value, the smoothed
`P(works | complaint)`, both scores, and the resulting `P(complaint)`.

Then say what happens at the two extremes of the sweep.

In [ ]:
print(f"{'alpha':>7}  {'P(works|comp)':>14}  {'P(complaint)':>13}  verdict")
for a in [0, 0.01, 0.1, 0.5, 1, 5, 20]:
    sb, sg = score(words, 1, a), score(words, 0, a)
    pw = (count("works", 1) + a) / (n(1) + 2 * a)
    print(f"{a:>7}  {pw:>14.4f}  {sb/(sb+sg):>13.4f}  "
          f"{'complaint' if sb > sg else 'compliment'}")

print()
print("the prior, for reference:", round(n(1) / n(), 4))

### Answer

```
  alpha   P(works|comp)   P(complaint)  verdict
      0          0.0000         0.0000  compliment
   0.01          0.0010         0.0191  compliment
    0.1          0.0098         0.1520  compliment
    0.5          0.0455         0.3947  compliment
      1          0.0833         0.4847  compliment
      5          0.2500         0.5313  complaint
     20          0.4000         0.4716  compliment
```

**At `alpha = 0`** the zero dominates and the answer is a certainty, 0.0000, decided
entirely by one unseen word.

**At `alpha = 1`** it reads 0.4847 — almost exactly undecided, which is the honest
answer for a three-word review with one word pointing each way.

**At `alpha = 20` something revealing happens: the number falls back towards 0.4167,
which is the prior.** We are adding 20 imaginary sightings of every word to a class
with only 10 real reviews, so the invented data drowns out the real data and every
likelihood is dragged towards 0.5. When all the likelihoods are equal they cancel, and
the model has nothing left to go on but the base rate. Push `alpha` higher and it
converges on the prior exactly — a model that has stopped reading the review.

**Watch the verdict column rather than just the number.** It says compliment, then
flips to complaint at `alpha = 5`, then flips *back* at `alpha = 20`. That wobble is
not the evidence changing its mind; it is the answer passing through 0.5 on its way
from "decided by one zero" to "decided by the prior". A verdict that depends this
sharply on a smoothing constant is a verdict you should not trust — and with a
three-word review and 24 training examples, that is the correct conclusion.

So `alpha` is a genuine bias–variance knob, like `max_depth` on Day 39 and `C` on
Day 41. Too small and the model trusts every accident in the data; too large and it
ignores the data entirely. The default of 1 is a reasonable middle, and when it matters
you tune it with `GridSearchCV` like any other hyperparameter.

## Question 10

Show the underflow problem for yourself, then show that logs fix it.

Multiply 0.01 by itself 150, 200 and 400 times and print each result. Then print the
sum of logs for the same cases. Explain why swapping to logs is allowed — what property
makes it safe?

In [ ]:
for k in [150, 200, 400]:
    product = 0.01 ** k
    logsum  = k * math.log(0.01)
    print(f"{k:>4} clues   product {product:>12.3e}   log-sum {logsum:>10.3f}"
          f"   underflowed: {product == 0.0}")

print()
print("smallest positive float64:", np.nextafter(0, 1))
print()
# the property that makes the swap safe: log is strictly increasing
a, b = 1e-120, 2e-120
print(f"a < b           : {a < b}")
print(f"log(a) < log(b) : {math.log(a) < math.log(b)}   <- order is preserved")

### Answer

At 150 clues the product is `1.000e-300`, still just representable. At **200 it is
exactly 0.0**, and at 400 it is still 0.0 — the number has fallen below the smallest
value a float64 can hold (about 5e-324) and been rounded away to nothing. Every class
would score zero and the comparison becomes meaningless.

The log-sums, meanwhile, are −690.8, −921.0 and −1842.1: perfectly ordinary numbers
that a computer handles without complaint. Taking a log has converted an impossible
range of magnitudes into a comfortable one.

**Why the swap is safe: `log` is strictly increasing.** If `a < b` then
`log(a) < log(b)`, always. Since classification only needs to know *which score is
bigger*, and logs preserve that ordering exactly, comparing log-scores gives the same
winner as comparing the original probabilities. Nothing is approximated and nothing is
lost.

The second benefit is that `log(x × y) = log(x) + log(y)`, so the product of 400
likelihoods becomes a sum of 400 logs — which is also faster and numerically steadier.

This is why sklearn exposes `predict_log_proba`, and why the textbook formula for naive
Bayes is written as a Σ of logs rather than a Π of probabilities.

---
# Part D · Choosing a flavour

## Question 11

Three datasets, three flavours. For each dataset below, run **all three** naive Bayes
flavours plus a random forest, and print the cross-validated accuracy.

- `load_digits()` — 64 columns of pixel **counts** (0–16)
- `load_breast_cancer()` — 30 columns of continuous **measurements**
- `load_wine()` — 13 columns of continuous **measurements** on very different scales

Then say which flavour won each one, and whether that is what the slides predicted.

In [ ]:
sets = {"digits": load_digits(return_X_y=True),
        "breast_cancer": load_breast_cancer(return_X_y=True),
        "wine": load_wine(return_X_y=True)}

for name, (X, y) in sets.items():
    print(f"\n{name}   ({X.shape[0]} rows x {X.shape[1]} columns)")
    results = {}
    for label, model in [("MultinomialNB", MultinomialNB()),
                         ("GaussianNB", GaussianNB()),
                         ("BernoulliNB", BernoulliNB()),
                         ("random forest", RandomForestClassifier(n_estimators=200,
                                                                  random_state=0))]:
        results[label] = cross_val_score(model, X, y, cv=CV).mean()
        print(f"   {label:<16} {results[label]:.4f}")
    nb_only = {k: v for k, v in results.items() if k != "random forest"}
    print(f"   -> best NB flavour: {max(nb_only, key=nb_only.get)}")

### Answer

```
digits          MultinomialNB 0.9009   GaussianNB 0.8508   BernoulliNB 0.8531   forest 0.9766
breast_cancer   MultinomialNB 0.8999   GaussianNB 0.9385   BernoulliNB 0.6274   forest 0.9667
wine            MultinomialNB 0.8483   GaussianNB 0.9719   BernoulliNB 0.3990   forest 0.9832
```

**Digits → MultinomialNB wins**, as predicted: the columns are pixel *counts*, which is
exactly what Multinomial is built for.

**Breast cancer and wine → GaussianNB wins**, also as predicted: both are continuous
measurements, so fitting a bell curve per class is the right model of a column.

**BernoulliNB is a disaster on both continuous sets**, and it is worth understanding
exactly why rather than just noting it. BernoulliNB *binarises* every column at a
threshold of 0 by default — it only ever asks "is this value above zero?" — and these
measurements are all positive.

On **wine**, all 13 columns become columns of pure ones. Every row looks identical to
the model, so it can do nothing but predict the most common class, and 0.3990 is
precisely that class's share of the data.

On **breast cancer**, 24 of the 30 columns go constant; the six that survive contain a
zero in only 13 of 569 rows, so they carry almost nothing. The result is the same — it
assigns *every single row* the same label, and 0.6274 is just the majority class.

Both numbers are the baseline wearing a model's name. Using the wrong flavour is not a
small penalty; it can erase the data completely while still returning a plausible-looking
accuracy.

The forest beats naive Bayes on all three, by between 1 and 8 points — the usual state
of affairs, and the honest reason naive Bayes is a baseline rather than a destination.

> Worth noticing: `wine` has columns on wildly different scales (proline is in the
> hundreds, others are near 1) and GaussianNB still scores 0.9719 **without any
> scaling** — because it fits each column its own curve and never compares columns to
> each other.

## Question 12

For each situation, say whether you would reach for **naive Bayes** or something else,
and give one sentence of justification.

1. 40,000 support emails to route to the right department, retrained every night.
2. An insurer needs the probability a claim is fraudulent, to decide how much to set aside.
3. A first look at a brand-new dataset at 4pm, with a meeting at 4:30.
4. 200 patients, 20,000 gene columns, published accuracy is what matters.
5. Predicting house price from square footage and number of bedrooms — two columns that move together closely.

### Answer

**1 — Naive Bayes.** Text routing is the home ground: thousands of word columns, a
multi-class target, and a nightly retrain that has to finish quickly. `MultinomialNB`
trains in one pass of counting, so the retrain is trivial, and text is the one domain
where it is genuinely competitive rather than merely fast.

**2 — Something else.** The insurer does not want a label, they want *the number* — and
Question 7 is the reason not to give them a naive Bayes number. Its probabilities are
pushed towards 0 and 1, so a reserve calculated from them would be systematically
wrong. Use logistic regression, which is naturally well calibrated, or calibrate a
forest with `CalibratedClassifierCV`.

**3 — Naive Bayes**, without hesitation. This is what a baseline is for. It costs ten
milliseconds, needs no scaling and essentially no tuning, and it tells you what the
floor looks like — so when something clever scores 0.91 later you know whether that is
good. Walking into the meeting with "naive Bayes gets 0.88" is far better than walking
in with nothing.

**4 — Naive Bayes is a reasonable choice**, which is the surprise on this list. With 200
rows and 20,000 columns almost everything overfits, and naive Bayes barely can: it
estimates each column independently and has one hyperparameter. That said, "published
accuracy matters" argues for trying a linear SVM alongside it (Day 41: small, wide data
is exactly the SVM's case) and reporting whichever wins on honest cross-validation.

**5 — Something else.** Two columns that move together closely is precisely the
condition the naive assumption denies, and with only two columns there is no safety in
numbers to rescue the ranking. Also, price is a *number*, not a class — this is a
regression problem, and naive Bayes is a classifier. Use linear regression from Day 34.

**The pattern.** Reach for naive Bayes when the columns are **many, cheap and roughly
independent** — above all text — or when you need an answer immediately. Look elsewhere
when you need a trustworthy probability, when there are few strongly-related columns, or
when the target is a number rather than a label.

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload
the notebook. Questions 3, 7, 10 and 12 are marked mostly on the explanation — a correct
number with no reasoning gets little credit.